# Actividad 3: Google Trends

Este cuaderno descarga las series y escribe `raw/gtrends.csv`. Todo lo que necesita está
en `data/BOL_GTrends_Keywords_2026-09-18.xlsx`.

**1. Un término por consulta, nunca en lote.**
Google escala los resultados **de 0 a 100 dentro de cada consulta**, así que un lote de cinco queda escalado respecto al término más buscado. Si se piden juntos `["dolar paralelo", "dolar hoy", "tipo de cambio", "comprar dolares", "trabajo"]`, `comprar dolares` sale con un máximo de **1**, y `prestamo` y `empleo` con **2**: enteros casi constantes, sin señal. Consultados uno por uno, cada término recorre el rango completo 0–100.

Por eso la columna `lote` del libro de términos existe pero **no se usa** aquí.

**2. Límites de consultas.**
Con 1 segundo de pausa, la consulta **42** seguida devuelve un error 429. Con 2 segundos pasan los 67 términos. Pero después de una corrida completa Google mantiene un bloqueo de **decenas de minutos**.
El cuaderno **guarda después de cada término** y **salta los que ya tiene**: si se interrumpe, se vuelve a correr (incluso al día siguiente) y sigue donde quedó.

In [ ]:
# ===========================================================================
# CARGA DE LA CONFIGURACIÓN  --  editar config.py, NO esta celda.
# Deja `cfg` (los parámetros del país) y `P` (las rutas estándar).
# ===========================================================================
import os, sys, importlib.util

CODE_DIR = os.getcwd()
if not os.path.exists(os.path.join(CODE_DIR, "config.py")):
    CODE_DIR = os.path.abspath(os.path.join(CODE_DIR, "..", "activity"))

_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config.py"))
cfg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)

PATH_RAW  = P["raw"]      # aquí se escribe gtrends.csv
PATH_DATA = P["data"]     # aquí está el libro de términos

print(f"[CONFIG] {cfg.COUNTRY} ({cfg.CODE})")
print(f"         raw  : {PATH_RAW}")
print(f"         data : {PATH_DATA}")

In [ ]:
# ===========================================================================
# Librerías y parámetros.
# trendspy se importa en la celda de descarga, no aquí: así la Parte B
# (descarga manual) funciona aunque trendspy no esté instalado.
# ===========================================================================
import re, time, random, unicodedata, warnings
from datetime import date

import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ---- parámetros ------------------------------------------------------------
GT_XLSX        = "BOL_GTrends_Keywords_2026-09-18.xlsx"   # en data/
GT_GEO         = cfg.CODE                # "BO" -- viene de config.py
GT_START       = "2004-01-01"            # Google Trends empieza aquí
GT_END         = date.today().strftime("%Y-%m-%d")
GT_TIMEFRAME   = f"{GT_START} {GT_END}"  # 2004 a hoy = "all" en la página web
GT_PAUSE       = 2.5                     # pausa típica entre consultas, en segundos; con 1.0 llega el 429
GT_PAUSE_MIN   = GT_PAUSE - 0.1          # cada término sortea su pausa entre estos dos límites,
GT_PAUSE_MAX   = GT_PAUSE + 0.5          # concentrada alrededor de GT_PAUSE
GT_MIN_NONZERO = 0.15                    # se descarta un término con menos meses distintos de cero
GT_REFRESH     = False                   # True = volver a bajar los términos que ya están en gtrends.csv
GT_CSV         = os.path.join(PATH_RAW, "gtrends.csv")


def slug(s):
    """'Producción de gas' -> 'produccion_de_gas'."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "_", s).strip("_")


print(f"geo={GT_GEO}  periodo={GT_TIMEFRAME}")

## Paso 1: lista de términos

In [ ]:
kw = pd.read_excel(os.path.join(PATH_DATA, GT_XLSX), sheet_name="keywords")

# La columna 'lote' se ignora a propósito -- ver la nota al inicio del cuaderno.
terms = (kw.loc[kw["location"] == GT_GEO, "keyword"]
           .dropna().drop_duplicates().tolist())

print(f"{len(terms)} términos distintos para geo={GT_GEO}")
kw.head(8)

## Paso 2: descarga con `trendspy`

In [ ]:
from trendspy import Trends              # si falla:  pip install trendspy

# ---- ¿qué hay ya? ----------------------------------------------------------
have = pd.DataFrame()
if os.path.exists(GT_CSV) and not GT_REFRESH:
    have = pd.read_csv(GT_CSV, index_col="date", parse_dates=["date"])
    print(f"archivo existente: {have.shape[1]} términos, "
          f"{have.index.min().date()} a {have.index.max().date()}")

    # Un archivo armado con un periodo MÁS CORTO no se retoma: sus términos
    # conservarían su fecha de inicio y los nuevos empezarían en GT_START,
    # dejando columnas que cubren periodos distintos.
    inicio_pedido = pd.Timestamp(GT_START)
    if have.index.min() > inicio_pedido + pd.Timedelta(days=45):
        print()
        print(f"  El archivo empieza en {have.index.min().date()} pero GT_START pide "
              f"{inicio_pedido.date()}.")
        print("  Se vuelve a descargar todo para que todas las series tengan el mismo")
        print("  periodo. (Para conservarlo, subir GT_START o poner GT_REFRESH = True.)")
        have = pd.DataFrame()

todo = [t for t in terms if f"gt_{slug(t)}" not in have.columns]
print(f"{len(todo)} término(s) por bajar, {len(terms) - len(todo)} ya presentes")
print()

tr = Trends(request_delay=GT_PAUSE)
series = {c: have[c] for c in have.columns}
thin, failed = [], []


def save():
    """Escribe lo que haya hasta ahora; se llama después de cada término."""
    if not series:
        return None
    df = pd.DataFrame(series)
    df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
    df.index.name = "date"
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df.to_csv(GT_CSV)
    return df


for i, term in enumerate(todo, 1):
    # pausa aleatoria para este término: trendspy la respeta antes de cada consulta
    tr.request_delay = random.triangular(GT_PAUSE_MIN, GT_PAUSE_MAX, GT_PAUSE)
    try:
        d = tr.interest_over_time([term], geo=GT_GEO, timeframe=GT_TIMEFRAME)
        col = [c for c in d.columns if c != "isPartial"][0]
        s = pd.to_numeric(d[col], errors="coerce")
        s.index = pd.to_datetime(d.index)
        share = float((s > 0).mean())

        if share < GT_MIN_NONZERO:
            thin.append(term)
            print(f"  [{i:2d}/{len(todo)}] {term:28} escaso ({share:.0%} distinto de cero) -- descartado")
        else:
            series[f"gt_{slug(term)}"] = s
            save()                                  # <- se guarda tras cada término
            print(f"  [{i:2d}/{len(todo)}] {term:28} ok  ({share:.0%} distinto de cero)")

    except Exception as e:
        failed.append(term)
        msg = str(e)
        print(f"  [{i:2d}/{len(todo)}] {term:28} !! {type(e).__name__}")
        if "429" in msg or "Too Many" in msg:
            print("      Límite de Google. Todo lo descargado hasta aquí ya está guardado.")
            print("      Esperar (decenas de minutos, a veces más) y volver a correr esta")
            print("      celda, que sigue desde aquí. O pasar a la Parte B.")
            break

gt = save()
print()
if gt is not None:
    print(f"gtrends.csv: {gt.shape[0]} filas x {gt.shape[1]} términos | "
          f"{gt.index.min().date()} a {gt.index.max().date()}")
if thin:
    print(f"demasiado escasos, no se guardan: {thin}")
if failed:
    print(f"no se bajaron en esta corrida: {failed}")

## Paso 3: visualización

In [ ]:
gt = pd.read_csv(GT_CSV, index_col="date", parse_dates=["date"]).sort_index()

# el último mes es parcial -- no es una caída de la actividad
gt_clean = gt.iloc[:-1]

show = [c for c in ["gt_dolar_paralelo", "gt_gasolina", "gt_cemento",
                    "gt_trabajo", "gt_inflacion"] if c in gt_clean.columns]

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
gt_clean[show].plot(ax=ax[0], lw=1)
ax[0].set_title(f"Google Trends, {cfg.COUNTRY} ({GT_GEO}) -- interés de búsqueda mensual")
ax[0].set_ylabel("0-100"); ax[0].legend(fontsize=8, ncol=3)

gt_clean[show].rolling(12).mean().plot(ax=ax[1], lw=1.2)
ax[1].set_ylabel("media de 12 meses"); ax[1].legend().remove()
plt.tight_layout(); plt.show()

print(f"{gt_clean.shape[1]} términos, {gt_clean.index.min().date()} a "
      f"{gt_clean.index.max().date()}")
print()
print("cobertura (proporción de meses mayores que cero), los 10 peores:")
cov = (gt_clean > 0).mean().sort_values()
print(cov.head(10).apply(lambda v: f"{v:.0%}").to_string())

---
# Parte B: descarga manual

`trendspy` es un cliente **no oficial**. Google limita la tasa de consultas, y después de
una corrida completa mantiene un bloqueo de **decenas de minutos** (a veces más).

Cuando eso pasa hay que descargar manualmente las series. La celda siguiente:

1. avisa qué términos faltan
2. **imprime en pantalla el enlace exacto** de Google Trends para cada uno.

Se hace clic, se descarga el CSV de la serie temporal (botón de bajar, arriba a la derecha
del gráfico), se deja el archivo en `raw/` **sin renombrarlo**, y se corre la última celda,
que arma el mismo `raw/gtrends.csv` que habría armado el raspador.

**También se puede hacer todo a mano:** correr la configuración y el Paso 1, saltar el
Paso 2 y seguir aquí. Si `gtrends.csv` no existe, la celda lista todos los términos.

In [ ]:
# ---- ¿qué falta? -----------------------------------------------------------
from urllib.parse import quote

gt_actual = pd.read_csv(GT_CSV, index_col="date", parse_dates=["date"]) \
            if os.path.exists(GT_CSV) else pd.DataFrame()

# 'thin' lo define la celda del raspado; esta celda tambien corre sola.
_thin = globals().get("thin", [])
pendientes = [x for x in terms
              if f"gt_{slug(x)}" not in gt_actual.columns and x not in _thin]


def enlace_trends(termino):
    """El mismo periodo y país que usa el raspador, para que las series coincidan.
    De 2004 a hoy es lo que la página llama 'all'."""
    periodo = "all" if GT_START == "2004-01-01" else f"{GT_START}%20{GT_END}"
    return ("https://trends.google.com/trends/explore"
            f"?date={periodo}&geo={GT_GEO}&q={quote(termino)}")


if not pendientes:
    print("Nada pendiente: el raspado trajo todos los términos.")
    print(f"{GT_CSV} tiene {gt_actual.shape[1]} series.")
else:
    print("=" * 70)
    print(f"FALTAN {len(pendientes)} DE {len(terms)} TÉRMINOS")
    print("=" * 70)
    print("Casi siempre es el límite de Google, no un error del código.")
    print("Dos opciones:")
    print("  1. Esperar (decenas de minutos) y volver a correr la celda del raspado.")
    print("     No se pierde nada: lo ya descargado está guardado.")
    print("  2. Descargarlos a mano con los enlaces de abajo y correr la última celda.")
    print()
    for x in pendientes:
        print(f"  {x}")
        print(f"    {enlace_trends(x)}")

    print()
    print("Abrir cada enlace, bajar el CSV de la serie temporal (boton de descarga,")
    print("arriba a la derecha del grafico) y dejar el archivo en raw/ SIN renombrarlo.")
    print("Despues correr la ultima celda de este cuaderno.")

## Parte final: construir `gtrends.csv`

Reconoce las dos formas en que Google exporta:

| archivo | forma |
|---|---|
| `multiTimeline.csv` | página antigua: una línea de título arriba, columnas `término: (País)` |
| `time_series_*.csv` | página actual: columnas `Time` y el término |

Cada columna se renombra `gt_<término>` con la misma regla que el raspador, así que un
término bajado a mano ocupa el mismo lugar que uno raspado. Lo descargado **se suma** a lo
que ya haya en `gtrends.csv`: añade las series nuevas y, si un término ya estaba, gana la
versión manual. Si `gtrends.csv` no existe, se crea.

In [ ]:
# ---- juntar los CSV bajados a mano -----------------------------------------
import glob

multi = glob.glob(os.path.join(PATH_RAW, "*ultiTimeline*.csv")) + \
        glob.glob(os.path.join(PATH_RAW, "*multi*.csv"))
unica = glob.glob(os.path.join(PATH_RAW, "time_series_*.csv"))
multi = sorted(set(multi))

print(f"{len(multi)} archivo(s) multi-término, {len(unica)} archivo(s) de serie única")


def _ordenar(dfi, col_fecha):
    dfi = dfi.rename(columns={col_fecha: "date"})
    dfi["date"] = pd.to_datetime(dfi["date"], errors="coerce")
    dfi = dfi.dropna(subset=["date"]).set_index("date")
    # "dólar paralelo: (Bolivia)" -> "gt_dolar_paralelo", igual que el raspador
    dfi.columns = ["gt_" + slug(re.sub(r":\s*\(.*\)\s*$", "", str(c)))
                   for c in dfi.columns]
    # Google escribe "<1" en los meses de interés mínimo: es casi cero, no un faltante
    return dfi.replace("<1", 0)


piezas = []
for f in multi:
    # Google pone una línea de título encima del encabezado real
    piezas.append(_ordenar(pd.read_csv(f, skiprows=1), "Month"))

for f in unica:
    piezas.append(_ordenar(pd.read_csv(f), "Time"))

if not piezas:
    print("No hay archivos manuales en raw/. Si el raspado ya funcionó, no hace falta.")
else:
    manual = pd.concat(piezas, axis=1)
    manual = manual.loc[:, ~manual.columns.duplicated()]
    manual = manual.apply(pd.to_numeric, errors="coerce").sort_index()
    manual.index.name = "date"

    # OJO: un cero es un mes tranquilo, no un dato faltante. No convertir a NA.

    if os.path.exists(GT_CSV):
        previo = pd.read_csv(GT_CSV, index_col="date", parse_dates=["date"])
        manual = manual.combine_first(previo)

    manual = manual.sort_index()
    manual.to_csv(GT_CSV)          # siempre raw/, nunca data/
    print(f"-> {GT_CSV}: {manual.shape[0]} filas x {manual.shape[1]} series | "
          f"{manual.index.min().date()} a {manual.index.max().date()}")

    # Los archivos descargados NO se borran: volver a bajarlos a mano cuesta
    # tiempo, y la versión anterior de este código los eliminaba.
    print("Los archivos originales se dejan en raw/ por si hay que revisarlos.")